# EV7 Taxi — ถามตอบภาษาไทยแบบไม่ต่อ API (เทรนเองในเครื่อง)

ถามเป็นภาษาไทย เช่น *"ตอนนี้รถคันไหนอยู่ภูเกตบ้าง"* → ได้คำตอบจากข้อมูลจริง **โดยไม่ส่งข้อมูลออกนอกเครื่อง**

## แนวคิด: แยกคำถามเป็นชิ้น แทนการให้ LLM คิดทั้งหมด

```
คำถาม ─► ① ดึงค่า (slot)      ทะเบียน · สถานที่ (ทนพิมพ์ผิด) · เวลา      ← กฎ + fuzzy match
       ─► ② จัดประเภทคำถาม   intent: หารถในพื้นที่ / สถานะรถ / ...    ← embedder + classifier (เทรนเอง)
       ─► ③ query ข้อมูลจริง  pandas บน points_clean.parquet            ← โค้ดที่เราเขียน · ตอบถูกทุกแถว
       ─► ④ สร้างคำตอบไทย      template                                ← ไม่แต่งเรื่อง
```

ส่วน ② ใช้ **backbone ภาษา** ตัวเดียวกับโมเดลจัดหมวดสินค้า ITEC (`paraphrase-multilingual-MiniLM-L12-v2` ที่ cache ไว้แล้ว) + เทรน classifier ด้านบนเอง — แนวเดียวกับ `scripts/itec/category`

ท้าย notebook (ส่วน 8) มีแบบ **ใช้ LLM โอเพนซอร์สรันเอง + fine-tune (LoRA)** สำหรับเมื่อคำถามหลากหลายเกินกว่า intent ที่กำหนด — ต้องมี GPU

> ⚠️ ผลลัพธ์มี **ทะเบียนรถและตำแหน่ง** (ข้อมูลส่วนบุคคล) · **ล้าง output (Clear All Outputs) ก่อน commit / ส่งต่อ**
> ข้อมูลเป็น batch (30 ก.ย. – 1 ต.ค. 2026) — คำว่า "ตอนนี้" = เวลาล่าสุดของข้อมูล ไม่ใช่เวลาจริง

แบบระบบรวม → `COM7-Knowledge-Base/06_Project/EV7 Taxi - Q&A Assistant Design.md`

## 1 · โหลดข้อมูล และเตรียมตาราง `pos` (จุด GPS + จังหวัด/อำเภอ + สถานะ)

In [ ]:
import json, re, math, random, difflib, time
from pathlib import Path
import numpy as np
import pandas as pd

TAXI = Path.cwd().parent if Path.cwd().name == "qa" else Path(r"C:\Projects\my-first-project\scripts\taxi")
OUT = TAXI / "_out"
MASTER = Path(r"D:\Master")

pts = pd.read_parquet(OUT / "points_clean.parquet",
                      columns=["vin", "plate", "meter", "signal", "lat", "lon", "speed", "direction", "gps_time", "dist"])
pts = pts.rename(columns={"vin": "car_id"})               # ⚠️ คีย์ประจำรถคือ VIN — car_id เดิมซ้ำกันได้ถึง 3 คัน
dist = pd.DataFrame([f["properties"] for f in json.load(open(OUT / "dist_pip.geojson", encoding="utf-8"))["features"]])
# คอลัมน์ dist ใน points = ลำดับ feature ใน dist_pip.geojson (สร้างโดย build_dashboard.assign_area)
pos = pts.assign(
    dist_code=dist.pc.str[2:].values[pts["dist"]],          # TH8301 → 8301
    prov_code=dist.ppc.str[2:].values[pts["dist"]],          # TH83 → 83
    dist_th=dist.th.values[pts["dist"]],
    prov_th=dist.pth.values[pts["dist"]],
    plate_key=pts.plate.str.replace(r"[\s-]", "", regex=True),   # ทอ-4141 / ทอ 4141 → ทอ4141
)
MOVING_KMH = 5
pos["status"] = np.select(
    [pos.signal.eq("V"), pos.meter.eq("F"), pos.speed.ge(MOVING_KMH)],
    ["ไม่มีสัญญาณ", "มีผู้โดยสาร", "ว่าง·วิ่ง"], default="จอด")
pos = pos.sort_values(["car_id", "gps_time"]).reset_index(drop=True)
LATEST = pos.gps_time.max()
print(f"{len(pos):,} จุด · {pos.car_id.nunique():,} คัน · ข้อมูลล่าสุด {LATEST}")
pos.head(3)

## 2 · รายชื่อสถานที่ (gazetteer) — จังหวัด/อำเภอ ไทย-อังกฤษ + ชื่อเล่น

In [ ]:
prov = dist[["ppc", "pth", "pen"]].drop_duplicates("ppc")
gaz = []                                                     # (ชื่อที่ใช้จับ, ระดับ, รหัส, ชื่อเต็มที่แสดง)
for r in prov.itertuples():
    for n in {r.pth, "จังหวัด" + r.pth, r.pen.lower()}:
        gaz.append((n, "prov", r.ppc[2:], r.pth))
for r in dist.itertuples():
    pre = "เขต" if r.ppc == "TH10" else "อำเภอ"             # "เขต" ใช้เฉพาะกรุงเทพฯ
    for n in {r.th, pre + r.th, r.en.lower()}:
        gaz.append((n, "dist", r.pc[2:], f"{r.th} ({r.pth})"))
ALIAS = {"กทม": "10", "กรุงเทพ": "10", "บางกอก": "10", "bkk": "10", "โคราช": "30", "อยุธยา": "14"}
name_of = dict(zip(prov.ppc.str[2:], prov.pth))
gaz += [(a, "prov", c, name_of[c]) for a, c in ALIAS.items()]
gaz = pd.DataFrame(gaz, columns=["name", "level", "code", "label"]).drop_duplicates("name")

# จุดกึ่งกลางจังหวัด (จาก D:\Master) ไว้หาสถานีชาร์จใกล้จังหวัด
cent = pd.read_csv(MASTER / "dim_province.csv", dtype={"province_code": str}, encoding="utf-8-sig") \
         .set_index("province_code")[["centroid_lat", "centroid_lon"]]
print(len(gaz), "ชื่อ ·", gaz.level.value_counts().to_dict())

## 3 · ดึงค่าจากคำถาม (slot) — ทะเบียน · เวลา · สถานที่

- **ทะเบียน:** regex รูปแบบ `ทอ-1234` / `ทอ 1234` / `E-1234`
- **เวลา:** คำว่า ตอนนี้/ล่าสุด · วันนี้ · เมื่อวาน · `HH:MM` หรือ `N โมง`
- **สถานที่:** เทียบทุกชื่อใน gazetteer กับทุกช่วงของคำถามด้วย `difflib` (ทนพิมพ์ผิด: "ภูเกต" ≈ "ภูเก็ต" = 0.91) · ตัดทะเบียน/เวลาออกก่อนกันจับผิด

In [ ]:
PLATE_RE = re.compile(r"(?<![ก-๙A-Za-z])((?:[ก-ฮ]{1,3}|[A-Za-z]{1,2})\s*-?\s*\d{3,4})(?!\d)")
TIME_RE = re.compile(r"(\d{1,2})\s*[:.]\s*(\d{2})\s*(?:น\.?)?|(\d{1,2})\s*โมง")

def find_plate(q):
    m = PLATE_RE.search(q)
    if not m:
        return None, q
    key = re.sub(r"[\s-]", "", m.group(1)).upper()
    return key, q.replace(m.group(1), " ")

def find_time(q):
    # คืน (เวลาเริ่ม, เวลาสิ้นสุด, คำอธิบาย) · ข้อมูลเป็น batch → อ้างอิงจาก LATEST
    day0 = LATEST.normalize()
    m = TIME_RE.search(q)
    if "เมื่อวาน" in q:
        a, b, label = day0 - pd.Timedelta(days=1), day0, "เมื่อวาน"
    elif "วันนี้" in q:
        a, b, label = day0, LATEST, "วันนี้"
    else:
        a, b, label = None, LATEST, "ล่าสุด"
    if m:
        hh = int(m.group(1) or m.group(3)); mm = int(m.group(2) or 0)
        base = a if a is not None else day0
        b = base + pd.Timedelta(hours=hh, minutes=mm)
        label = f"{label if a is not None else ''} {hh:02d}:{mm:02d} น.".strip()
        q = q.replace(m.group(0), " ")
    return (a, b, label), q

def grams(s):
    return {s[i:i + 2] for i in range(len(s) - 1)}
GAZ = [(r.name, len(r.name), grams(r.name), r) for r in gaz.itertuples() if len(r.name) >= 2]

def find_place(q, min_score=0.85):
    s = q.lower(); sg = grams(s)
    best = (0.0, None)
    for n, L, g, r in GAZ:
        if n in s:                                           # ตรงเป๊ะ: ชื่อยาวกว่าชนะ
            score = 1.0 + L / 100
        elif L < 4 or len(g & sg) < 0.4 * len(g):           # ชื่อสั้นต้องตรงเป๊ะ (fuzzy จับคำทั่วไปผิด) · กรองเร็วด้วยคู่ตัวอักษร
            continue
        else:
            score = max(difflib.SequenceMatcher(None, n, s[i:i + w]).ratio()   # ช่วงยาวใกล้เคียงชื่อ (พิมพ์ขาด/เกิน 1 ตัว)
                        for w in (L - 1, L, L + 1) for i in range(0, max(1, len(s) - w + 1)))
        if r.level == "prov":
            score += 0.001                                   # เสมอกัน → เลือกจังหวัด
        if score > best[0]:
            best = (score, r)
    if best[0] < min_score:
        return None
    r = best[1]
    return {"level": r.level, "code": r.code, "label": r.label, "matched": r.name, "score": round(min(best[0], 1.0), 2)}

def extract(q):
    plate, rest = find_plate(q)
    when, rest = find_time(rest)
    place = find_place(rest)
    return {"plate": plate, "when": when, "place": place}

for q in ["ตอนนี้รถคันไหนอยู่ภูเกตบ้าง", "ทอ-4141 อยู่ไหน", "เมื่อวาน ทอ 4141 ไปไหนมาบ้าง", "รถในเขตจตุจักรตอน 9 โมง"]:
    e = extract(q)
    print(q, "→", e["plate"], "|", e["when"][2], "|", e["place"] and (e["place"]["label"], e["place"]["score"]))

## 4 · สร้างชุดข้อมูลเทรน intent (สังเคราะห์จาก template)

ยังไม่มีคำถามจริงจากผู้ใช้ → สร้างจาก template หลายแบบต่อ intent แล้วสุ่มเติมค่า + ทำให้พิมพ์ผิดบางส่วน
**ทดสอบแบบแยก template** (template ที่ใช้ทดสอบไม่เคยเห็นตอนเทรน) — ถ้าสุ่มแบ่งธรรมดา คะแนนจะสูงเกินจริงเพราะประโยคเกือบซ้ำกัน

ก่อนเข้า classifier จะ **แทนค่าที่ดึงได้ด้วยคำกลาง** (`สถานที่` · `ทะเบียนรถ`) — โมเดลจะเรียนรูปประโยค ไม่ใช่ชื่อจังหวัด

In [ ]:
T = {
 "cars_in_area": [
   "ตอนนี้รถคันไหนอยู่{place}บ้าง", "มีรถกี่คันอยู่ที่{place}", "{place}มีรถอยู่กี่คัน", "รถที่อยู่ใน{place}ตอนนี้",
   "ขอรายชื่อรถใน{place}", "{place}ตอนนี้มีแท็กซี่ไหม", "แท็กซี่ใน{place}มีกี่คัน", "รถว่างใน{place}มีกี่คัน",
   "มีรถวิ่งแถว{place}บ้างไหม", "ใครอยู่{place}บ้าง", "เช็ครถในพื้นที่{place}", "cars in {place_en}",
   "รถคันไหนจอดอยู่{place}", "{place}มีรถมีผู้โดยสารกี่คัน", "ขอดูรถทั้งหมดที่อยู่{place}",
   "จำนวนรถที่อยู่ใน{place}", "{place}มีกี่คัน", "รถเราที่อยู่{place}ตอนนี้มีใครบ้าง", "list รถใน{place}"],
 "car_status": [
   "{plate} อยู่ไหน", "รถ {plate} ตอนนี้อยู่ที่ไหน", "ตำแหน่งล่าสุดของ {plate}", "{plate} วิ่งอยู่หรือจอด",
   "เช็คสถานะรถ {plate}", "ทะเบียน {plate} อยู่จังหวัดอะไร", "{plate} มีผู้โดยสารไหม", "where is {plate}",
   "รถ {plate} ตอนนี้เป็นยังไง", "{plate} ขับไปทางไหน", "หา {plate} ให้หน่อย", "{plate} อยู่แถวไหนแล้ว",
   "{plate} ไปถึงไหนแล้ว", "{plate} ตอนนี้อยู่จังหวัดไหน", "พิกัดรถ {plate}"],
 "car_track": [
   "{plate} ไปไหนมาบ้าง", "เส้นทางของ {plate} {time}", "{plate} วิ่งไปกี่กิโล{time}", "สรุปการเดินทางรถ {plate}",
   "{plate} ผ่านจังหวัดอะไรบ้าง", "ประวัติการวิ่งของ {plate}", "{time} {plate} ไปที่ไหนบ้าง", "{plate} วิ่งทั้งหมดกี่กิโลเมตร",
   "ย้อนดูเส้นทาง {plate}", "{plate} ออกต่างจังหวัดไหม{time}", "track {plate}"],
 "out_of_zone": [
   "{time}มีรถออกนอกเขตกี่คัน", "รถคันไหนออกนอกเขต", "ใครวิ่งออกนอกพื้นที่บ้าง", "รายการรถนอกเขต",
   "มีรถวิ่งออกต่างจังหวัดไหม", "รถที่ออกนอกกรุงเทพและปริมณฑล", "นับรถที่ออกนอกเขต{time}", "รถผิดพื้นที่มีกี่คัน",
   "มีใครขับออกนอกเขตที่อนุญาตไหม", "เช็ครถออกนอกโซน", "รถคันไหนวิ่งไปต่างจังหวัด{time}", "สรุปรถออกนอกพื้นที่{time}",
   "ใครออกนอกเขตบ้าง{time}", "มีรถหลุดโซนไหม"],
 "fleet_stats": [
   "จังหวัดไหนมีรถวิ่งเยอะที่สุด", "สรุปจำนวนรถแยกตามจังหวัด", "ตอนนี้รถออนไลน์กี่คัน", "รถมีผู้โดยสารกี่เปอร์เซ็นต์",
   "มีรถทั้งหมดกี่คัน", "พื้นที่ไหนรถเยอะสุด", "ภาพรวมรถตอนนี้", "จัดอันดับจังหวัดตามจำนวนรถ", "สถิติรถ{time}",
   "รถวิ่งอยู่กี่คันจอดกี่คัน", "ทั้งประเทศมีรถออนไลน์กี่คัน", "สรุปภาพรวมทั้งหมด{time}", "ตอนนี้มีรถวิ่งกี่คัน",
   "อัตรารถมีผู้โดยสารตอนนี้", "จังหวัดที่มีรถมากที่สุด 5 อันดับ"],
 "nearest_charger": [
   "{plate} ใกล้จุดชาร์จไหน", "สถานีชาร์จใกล้{place}", "จุดชาร์จ EV ที่ใกล้รถ {plate} ที่สุด", "หาที่ชาร์จแถว{place}",
   "แถว{place}มีปั๊มชาร์จไหม", "รถ {plate} ควรไปชาร์จที่ไหน", "ที่ชาร์จไฟใกล้ๆ{place}", "charger near {place_en}"],
 "action_request": [
   "ตัดสัญญาณรถ {plate} ให้หน่อย", "ล็อครถ {plate}", "สั่งดับเครื่อง {plate}", "ปิดรถ {plate} เลย",
   "ส่งข้อความหาคนขับ {plate}", "บล็อกรถคันนี้ {plate}", "หยุดรถ {plate}", "ล็อคล้อ {plate}", "ยึดรถ {plate}",
   "ตัดไฟรถ {plate}", "ปลดล็อครถ {plate}"],
 "other": [
   "สวัสดี", "อากาศวันนี้เป็นยังไง", "ช่วยเขียนอีเมลให้หน่อย", "ราคาหุ้นวันนี้", "คุณเป็นใคร", "ขอบคุณ",
   "เล่าเรื่องตลกหน่อย", "แปลภาษาอังกฤษให้หน่อย", "วันนี้วันอะไร", "กินอะไรดี", "ราคาน้ำมันวันนี้",
   "บอกข่าวล่าสุด", "ช่วยคิดเลขให้หน่อย", "เพลงเพราะๆ"],
}
TIMES = ["", "วันนี้", "เมื่อวาน", "ตอนนี้", "ตอน 9 โมง", "เมื่อ 14:30"]
plates = pos.plate.drop_duplicates().sample(300, random_state=1).tolist()
places = gaz[gaz.level == "prov"].sample(frac=1, random_state=1)
pen = dict(zip(prov.ppc.str[2:], prov.pen))

PRE = ["", "", "", "ขอ", "ช่วยบอกหน่อย ", "อยากรู้ว่า ", "รบกวนเช็ค ", "แอดมิน "]
POST = ["", "", "", " ครับ", " คะ", " หน่อย", " ด่วน", " ?", " นะ"]

def typo(s, p=0.15):
    # ทำให้พิมพ์ผิดเล็กน้อย: ลบ/สลับ/ซ้ำ 1 ตัวอักษร
    if random.random() > p or len(s) < 4:
        return s
    i = random.randrange(1, len(s) - 1); k = random.choice("dsr")
    return s[:i] + s[i + 1:] if k == "d" else s[:i] + s[i + 1] + s[i] + s[i + 2:] if k == "s" else s[:i] + s[i] + s[i:]

random.seed(7)
rows = []
for intent, tpls in T.items():
    for ti, t in enumerate(tpls):
        for _ in range(25):
            pl = places.sample(1, random_state=random.randrange(10**6)).iloc[0]
            q = t.format(place=typo(pl["name"]), place_en=pen.get(pl.code, pl["name"]),
                         plate=random.choice(plates), time=random.choice(TIMES))
            q = random.choice(PRE) + q + random.choice(POST)    # คำสุภาพ/คำเสริม ให้ประโยคหลากหลาย
            rows.append((re.sub(r"\s+", " ", q).strip(), intent, f"{intent}#{ti}"))
train_df = pd.DataFrame(rows, columns=["q", "intent", "tpl"]).drop_duplicates("q")

def mask(q):
    # แทนค่าที่ดึงได้ด้วยคำกลาง ก่อนส่งเข้า classifier
    e = extract(q)
    _, rest = find_plate(q)
    if e["plate"]:
        rest = rest + " ทะเบียนรถ"
    _, rest = find_time(rest)
    if e["place"]:
        rest = rest.lower().replace(e["place"]["matched"], " สถานที่ ")
        if "สถานที่" not in rest:
            rest += " สถานที่"
    return re.sub(r"\s+", " ", rest).strip()

train_df["masked"] = train_df.q.map(mask)
print(len(train_df), "ประโยค ·", train_df.intent.value_counts().to_dict())
train_df.sample(6, random_state=3)[["q", "masked", "intent"]]

## 5 · เทรน intent classifier — backbone เดียวกับโมเดล ITEC + Logistic Regression

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report

BACKBONE = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"   # ตัวเดียวกับ scripts/itec/category (cache ไว้แล้ว)
enc = SentenceTransformer(BACKBONE, device="cpu")

t0 = time.time()
X = enc.encode(train_df.masked.tolist(), batch_size=64, normalize_embeddings=True, show_progress_bar=False)
print(f"encode {len(X):,} ประโยค {time.time() - t0:.0f} วินาที")

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=0)       # แบ่งตาม template
tr, te = next(gss.split(X, train_df.intent, groups=train_df.tpl))
clf = LogisticRegression(C=8, max_iter=2000, class_weight="balanced").fit(X[tr], train_df.intent.iloc[tr])
print(classification_report(train_df.intent.iloc[te], clf.predict(X[te]), digits=3, zero_division=0))

In [ ]:
# เทรนใหม่ด้วยข้อมูลทั้งหมดสำหรับใช้งาน + ชุดทดสอบที่เขียนเอง (ไม่ได้มาจาก template)
clf = LogisticRegression(C=8, max_iter=2000, class_weight="balanced").fit(X, train_df.intent)

HAND = [
 ("ภูเก็ตตอนนี้มีรถเรามั้ย", "cars_in_area"), ("แถวบางนามีแท็กซี่ว่างกี่คันครับ", "cars_in_area"),
 ("ขอลิสต์รถที่อยู่เชียงใหม่", "cars_in_area"), ("นนทบุรีมีรถกี่คันตอนนี้", "cars_in_area"),
 ("ทอ-4141 ตอนนี้ไปถึงไหนแล้ว", "car_status"), ("รถทะเบียน ทอ 8675 จอดอยู่หรือเปล่า", "car_status"),
 ("ช่วยหาตำแหน่ง E-1234", "car_status"),
 ("เมื่อวานทอ-4141 วิ่งไปไหนบ้าง", "car_track"), ("ทอ-8571 วันนี้วิ่งไปกี่โล", "car_track"),
 ("วันนี้มีใครออกนอกเขตบ้าง", "out_of_zone"), ("รถที่วิ่งออกต่างจังหวัดวันนี้", "out_of_zone"),
 ("ตอนนี้รถวิ่งอยู่ทั้งหมดกี่คัน", "fleet_stats"), ("จังหวัดไหนรถเยอะที่สุดตอนนี้", "fleet_stats"),
 ("ทอ-4141 แบตใกล้หมด ชาร์จที่ไหนใกล้สุด", "nearest_charger"), ("ที่ชาร์จรถไฟฟ้าแถวหาดใหญ่", "nearest_charger"),
 ("ช่วยตัดไฟรถ ทอ-4141 ที", "action_request"), ("ล็อคล้อรถคันนี้ ทอ-8675", "action_request"),
 ("หวัดดีครับ", "other"), ("ฝนจะตกไหมวันนี้", "other"), ("ขอสูตรต้มยำ", "other"),
]
hand = pd.DataFrame(HAND, columns=["q", "intent"])
Xh = enc.encode(hand.q.map(mask).tolist(), normalize_embeddings=True)
proba = clf.predict_proba(Xh)
hand["pred"] = clf.classes_[proba.argmax(1)]
hand["conf"] = proba.max(1).round(2)
print(f"ถูก {(hand.pred == hand.intent).mean():.0%} ของ {len(hand)} ข้อ")
hand[hand.pred != hand.intent]

## 6 · ตัวทำงาน (executor) — query ข้อมูลจริง

ทุกตัวอ่านอย่างเดียว · ผลลัพธ์คืนเป็น dict ให้ส่วนสร้างคำตอบใช้ (และให้หน้าเว็บไฮไลต์รถได้)

In [ ]:
STALE = pd.Timedelta(minutes=15)
ZONE = {"10", "11", "12", "13", "73", "74"}             # กรุงเทพฯ และปริมณฑล
chargers = json.load(open(OUT / "ev_chargers_osm.json", encoding="utf-8"))
chg = pd.DataFrame([{"lat": (e.get("lat") or e["center"]["lat"]), "lon": (e.get("lon") or e["center"]["lon"]),
                     "name": e.get("tags", {}).get("name", "") or "(ไม่มีชื่อ)",
                     "op": e.get("tags", {}).get("operator") or e.get("tags", {}).get("brand") or ""}
                    for e in chargers if "lat" in e or "center" in e])

def hav_km(lat1, lon1, lat2, lon2):
    p = np.pi / 180
    a = np.sin((lat2 - lat1) * p / 2) ** 2 + np.cos(lat1 * p) * np.cos(lat2 * p) * np.sin((lon2 - lon1) * p / 2) ** 2
    return 12742 * np.arcsin(np.sqrt(a))

def latest_at(t):
    w = pos[(pos.gps_time <= t) & (pos.gps_time > t - STALE)]
    return w.groupby("car_id").tail(1)

def car_rows(plate_key):
    return pos[pos.plate_key.str.upper() == plate_key]

def x_cars_in_area(place, when):
    if not place:
        return {"error": "ไม่พบชื่อสถานที่ในคำถาม"}
    now = latest_at(when[1])
    col = "prov_code" if place["level"] == "prov" else "dist_code"
    hit = now[now[col] == place["code"]]
    return {"as_of": when[1], "place": place["label"], "n": len(hit), "by_status": hit.status.value_counts().to_dict(),
            "cars": hit[["car_id", "plate", "status", "dist_th"]].head(20).to_dict("records")}

def x_car_status(plate, when):
    r = car_rows(plate) if plate else pd.DataFrame()
    if r.empty:
        return {"error": f"ไม่พบรถทะเบียน {plate or '(ไม่ได้ระบุ)'}"}
    r = r[r.gps_time <= when[1]]
    if r.empty:
        return {"error": "ไม่มีข้อมูลก่อนเวลาที่ถาม"}
    x = r.iloc[-1]
    return {"as_of": when[1], "plate": x.plate, "car_id": x.car_id, "time": x.gps_time, "age_min": round((when[1] - x.gps_time).total_seconds() / 60),
            "status": x.status, "speed": x.speed, "heading": x.direction, "where": f"{x.dist_th}, {x.prov_th}", "lat": x.lat, "lon": x.lon}

def x_car_track(plate, when):
    r = car_rows(plate) if plate else pd.DataFrame()
    if r.empty:
        return {"error": f"ไม่พบรถทะเบียน {plate or '(ไม่ได้ระบุ)'}"}
    a = when[0] if when[0] is not None else LATEST.normalize()
    r = r[(r.gps_time >= a) & (r.gps_time <= when[1]) & (r.signal != "V")]
    if len(r) < 2:
        return {"error": "ข้อมูลในช่วงนี้ไม่พอ"}
    gap = r.gps_time.diff() <= STALE                       # ไม่นับระยะข้ามช่วงที่สัญญาณหาย
    km = hav_km(r.lat.shift(), r.lon.shift(), r.lat, r.lon)[gap].sum()
    provs = r.groupby("prov_th", sort=False).size()
    return {"plate": r.plate.iloc[0], "from": r.gps_time.iloc[0], "to": r.gps_time.iloc[-1], "km": round(float(km), 1),
            "max_speed": r.speed.max(), "provinces": list(provs.index), "start": f"{r.dist_th.iloc[0]}, {r.prov_th.iloc[0]}",
            "end": f"{r.dist_th.iloc[-1]}, {r.prov_th.iloc[-1]}"}

def x_out_of_zone(when):
    a = when[0] if when[0] is not None else LATEST.normalize()
    r = pos[(pos.gps_time >= a) & (pos.gps_time <= when[1]) & (pos.signal != "V") & ~pos.prov_code.isin(ZONE)]
    g = r.groupby(["car_id", "plate"]).agg(provinces=("prov_th", lambda s: ", ".join(s.unique()[:3])), points=("prov_th", "size"))
    g = g.sort_values("points", ascending=False).reset_index()
    return {"from": a, "to": when[1], "n": len(g), "cars": g.head(20).to_dict("records")}

def x_fleet_stats(when):
    now = latest_at(when[1])
    top = now.groupby("prov_th").size().sort_values(ascending=False).head(5)
    return {"as_of": when[1], "online": len(now), "by_status": now.status.value_counts().to_dict(), "top_prov": top.to_dict()}

def x_nearest_charger(plate, place, when, k=3):
    if plate:
        s = x_car_status(plate, when)
        if "error" in s:
            return s
        lat, lon, ref = s["lat"], s["lon"], f"รถ {s['plate']} ({s['where']})"
    elif place:
        code = place["code"][:2]
        lat, lon = cent.loc[code, "centroid_lat"], cent.loc[code, "centroid_lon"]
        ref = f"กลางจังหวัด {name_of[code]}"
    else:
        return {"error": "ต้องระบุทะเบียนรถหรือสถานที่"}
    d = chg.assign(km=hav_km(lat, lon, chg.lat, chg.lon)).nsmallest(k, "km")
    return {"ref": ref, "items": d[["name", "op", "km"]].round(1).to_dict("records"),
            "note": "ข้อมูลสถานีชาร์จจาก OpenStreetMap มีแค่บางส่วน (306 จุด)"}

## 7 · รวมเป็น `ask()` — ดึงค่า → จัดประเภท → query → ตอบภาษาไทย

In [ ]:
MIN_CONF = 0.45            # มั่นใจต่ำกว่านี้ = ถามกลับ ไม่เดา

def fmt_t(t):
    return pd.Timestamp(t).strftime("%d/%m %H:%M")

def answer(intent, r):
    if "error" in r:
        return f"ขออภัย {r['error']}"
    if intent == "cars_in_area":
        if not r["n"]:
            return f"ไม่มีรถออนไลน์ใน{r['place']} (ข้อมูล ณ {fmt_t(r['as_of'])})"
        st = " · ".join(f"{k} {v}" for k, v in r["by_status"].items())
        lst = ", ".join(c["plate"] for c in r["cars"][:10])
        more = f" และอีก {r['n'] - 10} คัน" if r["n"] > 10 else ""
        return f"{r['place']} มีรถออนไลน์ {r['n']} คัน ({st}) — {lst}{more} · ข้อมูล ณ {fmt_t(r['as_of'])}"
    if intent == "car_status":
        return (f"{r['plate']} อยู่ที่ {r['where']} · {r['status']} {r['speed']:.0f} กม./ชม. ทิศ {r['heading']:.0f}° "
                f"· จุดล่าสุด {fmt_t(r['time'])} ({r['age_min']} นาทีก่อนเวลาที่ถาม)")
    if intent == "car_track":
        return (f"{r['plate']} ช่วง {fmt_t(r['from'])}–{fmt_t(r['to'])} วิ่งประมาณ {r['km']} กม. (เร็วสุด {r['max_speed']:.0f} กม./ชม.) "
                f"จาก {r['start']} ถึง {r['end']} · ผ่าน {', '.join(r['provinces'])}")
    if intent == "out_of_zone":
        lst = ", ".join(f"{c['plate']} ({c['provinces']})" for c in r["cars"][:8])
        return f"ช่วง {fmt_t(r['from'])}–{fmt_t(r['to'])} มีรถออกนอกกรุงเทพฯ และปริมณฑล {r['n']} คัน" + (f" — {lst}" if lst else "")
    if intent == "fleet_stats":
        st = " · ".join(f"{k} {v}" for k, v in r["by_status"].items())
        top = ", ".join(f"{k} {v}" for k, v in r["top_prov"].items())
        return f"ณ {fmt_t(r['as_of'])} รถออนไลน์ {r['online']:,} คัน ({st}) · จังหวัดที่รถเยอะสุด: {top}"
    if intent == "nearest_charger":
        lst = " · ".join(f"{i['name']}{' (' + i['op'] + ')' if i['op'] else ''} {i['km']} กม." for i in r["items"])
        return f"สถานีชาร์จใกล้{r['ref']}: {lst} — {r['note']}"
    return ""

def ask(q, verbose=False):
    e = extract(q)
    p = clf.predict_proba(enc.encode([mask(q)], normalize_embeddings=True))[0]
    intent, conf = clf.classes_[p.argmax()], float(p.max())
    if verbose:
        print(f"  intent={intent} ({conf:.2f}) plate={e['plate']} place={e['place'] and e['place']['label']} when={e['when'][2]}")
    if conf < MIN_CONF:
        return "ไม่แน่ใจว่าถามเรื่องอะไร ลองถามเช่น 'ตอนนี้รถคันไหนอยู่ภูเก็ต' หรือ 'ทอ-4141 อยู่ไหน'"
    if intent == "action_request":
        return "ระบบนี้ตอบคำถามได้อย่างเดียว ไม่สามารถสั่งการรถ (ตัดสัญญาณ/ล็อก/ดับเครื่อง) ได้"
    if intent == "other":
        return "ตอบได้เฉพาะคำถามเรื่องรถแท็กซี่ EV7 เช่น ตำแหน่งรถ รถในพื้นที่ เส้นทาง รถนอกเขต สถานีชาร์จ"
    run = {"cars_in_area": lambda: x_cars_in_area(e["place"], e["when"]),
           "car_status": lambda: x_car_status(e["plate"], e["when"]),
           "car_track": lambda: x_car_track(e["plate"], e["when"]),
           "out_of_zone": lambda: x_out_of_zone(e["when"]),
           "fleet_stats": lambda: x_fleet_stats(e["when"]),
           "nearest_charger": lambda: x_nearest_charger(e["plate"], e["place"], e["when"])}[intent]
    return answer(intent, run())

demo_plate = pos.plate.value_counts().index[0]      # ทะเบียนที่มีข้อมูลมากสุด ใช้ทดลอง
for q in ["ตอนนี้รถคันไหนอยู่ภูเกตบ้าง", f"{demo_plate} อยู่ไหน", f"วันนี้ {demo_plate} ไปไหนมาบ้าง",
          "วันนี้มีรถออกนอกเขตกี่คัน", "จังหวัดไหนมีรถเยอะที่สุด", "ที่ชาร์จแถวเชียงใหม่",
          f"ตัดสัญญาณรถ {demo_plate} ให้หน่อย", "ขอสูตรต้มยำ"]:
    print("Q:", q); print("A:", ask(q, verbose=True)); print()

In [ ]:
# บันทึกโมเดล (backbone ไม่ต้องบันทึก — โหลดจากชื่อ)
import joblib
joblib.dump({"clf": clf, "backbone": BACKBONE, "min_conf": MIN_CONF, "trained": str(pd.Timestamp.now())},
            OUT / "qa_intent.joblib")
print("บันทึก", OUT / "qa_intent.joblib")

## 8 · ทางเลือก: LLM โอเพนซอร์สรันเอง + fine-tune (ต้องมี GPU)

ส่วน 1–7 ตอบได้เฉพาะ **8 ประเภทคำถามที่กำหนด** · ถ้าผู้ใช้ถามหลากหลายกว่านี้ (เงื่อนไขซ้อน เปรียบเทียบ ถามต่อเนื่อง) ค่อยใช้ LLM รันเอง

| | intent + slot (ส่วน 1–7) | LLM รันเอง (ไม่ fine-tune) | LLM รันเอง + fine-tune (LoRA) | API (Claude) |
|---|---|---|---|---|
| ข้อมูลออกนอกองค์กร | ไม่ออก | ไม่ออก | ไม่ออก | ออก (ต้องขอ DPO) |
| ฮาร์ดแวร์ | CPU ธรรมดา | GPU ≥ 16–24 GB (รุ่น 7–8B แบบ 4-bit) | GPU ตอนเทรน + ตอนใช้ | ไม่ต้อง |
| คำถามนอกแบบ | ตอบไม่ได้ (ถามกลับ) | พอได้ | ดีขึ้นในโดเมนเรา | ดีที่สุด |
| ภาษาไทย | ดี (backbone multilingual) | ขึ้นกับรุ่น — เลือกรุ่นที่ฝึกภาษาไทย | ดีขึ้นด้วยข้อมูลเรา | ดี |
| งานดูแล | น้อย | กลาง (server GPU) | สูง (เทรนซ้ำ · ประเมินผล) | น้อย |
| ค่าใช้จ่าย | ~0 | ค่าเครื่อง GPU ตลอดเวลาที่เปิด | + ค่าเทรน | ต่อคำถาม |

**รุ่นที่น่าลอง** (ตรวจ license ก่อนใช้เชิงพาณิชย์): โมเดลที่ฝึกภาษาไทยเพิ่ม เช่น **Typhoon** (SCB 10X) · **OpenThaiGPT** · หรือรุ่นหลายภาษาขนาด 7–8B เช่น **Qwen2.5-Instruct** `[อนุมาน — ชื่อรุ่นและเงื่อนไข license ต้องตรวจจาก Hugging Face ณ วันที่ใช้]`

**วิธี fine-tune ที่เหมาะกับงานนี้:** ไม่ต้องสอนให้ LLM "รู้ตำแหน่งรถ" (ข้อมูลเปลี่ยนทุกนาที) — สอนให้ **แปลงคำถาม → คำสั่ง JSON** แล้วโค้ดในส่วน 6 เป็นคนดึงข้อมูลเหมือนเดิม
ชุดเทรนใช้ `train_df` จากส่วน 4 + คำถามจริงที่สะสม เช่น
`"ภูเกตมีรถกี่คัน" → {"intent": "cars_in_area", "place": "ภูเก็ต", "time": "latest"}`

cell ข้างล่าง **ไม่รันโดยค่าเริ่มต้น** (`RUN_LLM = False`) — ต้องติดตั้ง `peft` `bitsandbytes` และมี GPU

In [ ]:
RUN_LLM = False          # เปลี่ยนเป็น True บนเครื่องที่มี GPU (เช่น EC2 g5/g6) เท่านั้น
LLM_ID = "<ชื่อรุ่นบน Hugging Face เช่น typhoon / Qwen2.5-7B-Instruct>"   # เลือกและตรวจ license ก่อน

def to_target(q, intent):
    # คำตอบที่อยากให้ LLM สร้าง = JSON คำสั่ง (ไม่ใช่คำตอบสุดท้าย)
    e = extract(q)
    return json.dumps({"intent": intent, "plate": e["plate"], "place": e["place"] and e["place"]["label"],
                       "time": e["when"][2]}, ensure_ascii=False)

sft = [{"prompt": f"แปลงคำถามเป็นคำสั่ง JSON\nคำถาม: {r.q}\nJSON:", "completion": " " + to_target(r.q, r.intent)}
       for r in train_df.sample(min(2000, len(train_df)), random_state=0).itertuples()]
print(len(sft), "ตัวอย่าง ·", sft[0])

if RUN_LLM:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model
    tok = AutoTokenizer.from_pretrained(LLM_ID)
    model = AutoModelForCausalLM.from_pretrained(
        LLM_ID, device_map="auto",
        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16))
    model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                                             target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], task_type="CAUSAL_LM"))
    model.print_trainable_parameters()
    # ต่อด้วย training loop (เช่น transformers Trainer / TRL SFTTrainer) บน `sft`
    # ประเมิน: ให้โมเดลสร้าง JSON ของชุด HAND (ส่วน 5) แล้วเทียบ intent/slot กับคำตอบที่ถูก
    # ใช้งาน: LLM สร้าง JSON → ตรวจ schema → ส่งเข้า executor ส่วน 6 → answer() ส่วน 7

## 9 · ขั้นต่อไป

1. **เก็บคำถามจริงจากผู้ใช้** (เริ่มจาก 50–100 ข้อ) → เพิ่มใน `HAND` เป็นชุดทดสอบ · ข้อที่ผิดเอาไปเพิ่ม template / เทรนใหม่
2. ต่อ `ask()` เป็น API เล็ก ๆ (FastAPI) แล้วทำกล่องแชตในหน้าแผนที่ — คืน `car_id` ให้หน้าเว็บไฮไลต์รถ
3. ถ้าคำถามนอก 8 ประเภทเยอะ → ส่วน 8 (LLM รันเอง) หรือ API ตาม `EV7 Taxi - Q&A Assistant Design.md`